# Parcel Attributes ETL — Development & Testing

Use this notebook to explore schemas, prototype new overlay joins, and validate
proximity relationship logic before promoting changes to `Parcel_Attributes_ETL.py`.

**Target layer:** https://maps.trpa.org/server/rest/services/Parcel_Joins/FeatureServer/6

In [ ]:
from pathlib import Path
import os, json, requests
import pandas as pd
import arcpy
from arcgis.features import FeatureLayer

# Paths
SDE_DIR   = Path(r'F:\GIS\DB_CONNECT')
VECTOR_SDE    = str(SDE_DIR / 'Vector.sde')
COLLECTION_SDE = str(SDE_DIR / 'Collection.sde')
SCRATCH_GDB   = Path(r'C:\GIS\Scratch.gdb')

TARGET_FC     = os.path.join(COLLECTION_SDE, 'SDE.Parcel_Permit_Review_Attributes')
PARCEL_MASTER = os.path.join(VECTOR_SDE, r'SDE.Parcels\SDE.Parcel_Master')

arcpy.env.workspace        = str(SCRATCH_GDB)
arcpy.env.overwriteOutput  = True

# Web service
TARGET_SERVICE_URL = 'https://maps.trpa.org/server/rest/services/Parcel_Joins/FeatureServer/6'

print('Config OK')

## 1. Schema comparison — Target table vs Parcel Master

In [ ]:
# ── Target table schema from web service ──────────────────────────────────────
resp = requests.get(TARGET_SERVICE_URL, params={'f': 'json'})
svc  = resp.json()

target_fields = pd.DataFrame(svc['fields'])[['name', 'alias', 'type', 'length']]
print(f"Target table: {svc.get('name')}  ({len(target_fields)} fields)")
target_fields

In [ ]:
# ── Parcel Master schema from SDE ─────────────────────────────────────────────
pm_fields = pd.DataFrame(
    [{'name': f.name, 'alias': f.aliasName, 'type': f.type, 'length': f.length}
     for f in arcpy.ListFields(PARCEL_MASTER)]
)
print(f"Parcel Master: {len(pm_fields)} fields")
pm_fields

In [ ]:
# ── Which target fields exist in Parcel Master? (confirms direct-copy candidates)
target_names = set(target_fields['name'].str.upper())
pm_names     = set(pm_fields['name'].str.upper())

in_both   = sorted(target_names & pm_names)
only_target = sorted(target_names - pm_names)

print(f'Fields in BOTH target and Parcel Master ({len(in_both)}):')  
print(' ', '\n  '.join(in_both))
print()
print(f'Fields only in target (need spatial join or derivation) ({len(only_target)}):')
print(' ', '\n  '.join(only_target))

## 2. Prototype — Proximity spatial relationship (Within / Intersects / Outside)

In [ ]:
# Test the proximity logic against a small sample parcel set and one overlay.
# Replace OVERLAY_FC with any proximity overlay path to test.

JOIN_KEY   = 'APN'
OVERLAY_FC = os.path.join(VECTOR_SDE, r'SDE.Water\SDE.HWL_300ftBuffer')
ATTR_FIELD = None   # set to a field name like 'FLD_ZONE' to get "AE - Within" style values

# Use a small sample to keep the test fast
SAMPLE_FC = str(SCRATCH_GDB / 'test_parcels_sample')
if not arcpy.Exists(SAMPLE_FC):
    arcpy.analysis.Select(
        PARCEL_MASTER, SAMPLE_FC,
        where_clause="JURISDICTION = 'City of South Lake Tahoe'"
    )
print(f'Sample parcels: {int(arcpy.management.GetCount(SAMPLE_FC)[0]):,}')

In [ ]:
# Pass 1: parcels COMPLETELY WITHIN the overlay
within_fc = str(SCRATCH_GDB / 'test_prox_within')
arcpy.analysis.SpatialJoin(
    target_features=SAMPLE_FC,
    join_features=OVERLAY_FC,
    out_feature_class=within_fc,
    join_operation='JOIN_ONE_TO_ONE',
    join_type='KEEP_ALL',
    match_option='COMPLETELY_WITHIN',
)
within_apns = set()
with arcpy.da.SearchCursor(within_fc, [JOIN_KEY, 'Join_Count']) as cur:
    for apn, cnt in cur:
        if apn and cnt and cnt > 0:
            within_apns.add(apn)
print(f'Within: {len(within_apns):,}')

In [ ]:
# Pass 2: parcels that INTERSECT the overlay (any overlap)
intersect_fc = str(SCRATCH_GDB / 'test_prox_intersect')
read_i = [JOIN_KEY, 'Join_Count'] + ([ATTR_FIELD] if ATTR_FIELD else [])
arcpy.analysis.SpatialJoin(
    target_features=SAMPLE_FC,
    join_features=OVERLAY_FC,
    out_feature_class=intersect_fc,
    join_operation='JOIN_ONE_TO_ONE',
    join_type='KEEP_ALL',
    match_option='INTERSECT',
)
intersect_apns = {}
with arcpy.da.SearchCursor(intersect_fc, read_i) as cur:
    for row in cur:
        apn, cnt = row[0], row[1]
        if apn and cnt and cnt > 0:
            intersect_apns[apn] = row[2] if ATTR_FIELD else None
print(f'Intersects: {len(intersect_apns):,}')

In [ ]:
# Combine results
rows = []
with arcpy.da.SearchCursor(SAMPLE_FC, [JOIN_KEY]) as cur:
    for (apn,) in cur:
        if apn is None:
            continue
        if apn in within_apns:
            label = 'Within'
        elif apn in intersect_apns:
            label = 'Intersects'
        else:
            label = 'Outside'
        if ATTR_FIELD and apn in intersect_apns:
            attr_val = intersect_apns[apn]
            if attr_val and str(attr_val).strip():
                label = f"{str(attr_val).strip()} - {label}"
        rows.append({'APN': apn, 'RELATIONSHIP': label})

df = pd.DataFrame(rows)
print(df['RELATIONSHIP'].value_counts())
df.head(20)

## 3. Incremental update trigger — last_edited_date comparison

In [ ]:
def get_last_edited_date(fc, field='last_edited_date'):
    """Return the most-recent value of *field* in *fc*."""
    fields = [f.name for f in arcpy.ListFields(fc)]
    if field not in fields:
        print(f'  WARNING: field "{field}" not found in {fc}')
        return None
    with arcpy.da.SearchCursor(
        fc, [field], sql_clause=(None, f'ORDER BY {field} DESC')
    ) as cur:
        for row in cur:
            return row[0]
    return None

pm_date     = get_last_edited_date(PARCEL_MASTER)
target_date = get_last_edited_date(TARGET_FC)

print(f'Parcel Master last edited : {pm_date}')
print(f'Target table last edited  : {target_date}')
print()
if pm_date and target_date:
    if pm_date > target_date:
        print('→ UPDATE NEEDED')
    else:
        print('→ Target is up to date — no ETL run required')
else:
    print('→ Cannot determine — would default to running the ETL')

## 4. Verify overlay layer paths

In [ ]:
# Check which overlay paths actually exist in the SDE.
# Update paths in the ETL script for any that show MISSING.

overlays_to_check = {
    'Parcel Master':          PARCEL_MASTER,
    'Target FC':              TARGET_FC,
    'HRA':                    os.path.join(VECTOR_SDE, r'SDE.Water\SDE.Hydro_Areas'),
    'NRCS Soils 2003':        os.path.join(VECTOR_SDE, r'SDE.Soils\SDE.NRCS_Soils_2003'),
    'Priority Watersheds':    os.path.join(VECTOR_SDE, r'SDE.Water\SDE.Priority_Watersheds'),
    'FEMA Flood Zone':        os.path.join(VECTOR_SDE, r'SDE.Water\SDE.FEMA_Flood_Zone'),
    'Avalanche Zones':        os.path.join(VECTOR_SDE, r'SDE.Physiography\SDE.Avalanche_Zones'),
    'HWL 300ft Buffer':       os.path.join(VECTOR_SDE, r'SDE.Water\SDE.HWL_300ftBuffer'),
    'Town Center':            os.path.join(VECTOR_SDE, r'SDE.Planning\SDE.TownCenter'),
    'Zoning District':        os.path.join(VECTOR_SDE, r'SDE.Planning\SDE.District'),
    'Conservation/Rec':       os.path.join(VECTOR_SDE, r'SDE.Planning\SDE.Conservation_Rec'),
    'Historic Districts':     os.path.join(VECTOR_SDE, r'SDE.Historic\SDE.HistoricDistricts'),
    'Scenic Corridor':        os.path.join(VECTOR_SDE, r'SDE.Scenic\SDE.Scenic_Corridor'),
}

rows = []
for name, path in overlays_to_check.items():
    exists = arcpy.Exists(path)
    count  = int(arcpy.management.GetCount(path)[0]) if exists else None
    rows.append({'Layer': name, 'Exists': '✓' if exists else 'MISSING', 'Count': count, 'Path': path})

pd.DataFrame(rows)

In [ ]:
overlays_to_check = {
    'Parcel Master':          PARCEL_MASTER,
    'Target FC':              TARGET_FC,
    'HRA':                    os.path.join(VECTOR_SDE, r'SDE.Water\SDE.Hydro_Areas'),
    'NRCS Soils 2003':        os.path.join(VECTOR_SDE, r'SDE.Soils\SDE.NRCS_Soils_2003'),
    'Priority Watersheds':    os.path.join(VECTOR_SDE, r'SDE.Water\SDE.Priority_Watersheds'),
    'FEMA Flood Zone':        os.path.join(VECTOR_SDE, r'SDE.Water\SDE.FEMA_Flood_Zone'),
    'Avalanche Zones':        os.path.join(VECTOR_SDE, r'SDE.Physiography\SDE.Avalanche_Zones'),
    'HWL 300ft Buffer':       os.path.join(VECTOR_SDE, r'SDE.Water\SDE.HWL_300ftBuffer'),
    'Town Center':            os.path.join(VECTOR_SDE, r'SDE.Planning\SDE.TownCenter'),
    'Zoning District':        os.path.join(VECTOR_SDE, r'SDE.Planning\SDE.District'),
    'Conservation/Rec':       os.path.join(VECTOR_SDE, r'SDE.Planning\SDE.Conservation_Rec'),
    'Historic Districts':     os.path.join(VECTOR_SDE, r'SDE.Historic\SDE.HistoricDistricts'),
    'Scenic Corridor':        os.path.join(VECTOR_SDE, r'SDE.Scenic\SDE.Scenic_Corridor'),
}


In [ ]:
# Inspect field names for a specific overlay to verify FIELD_MAP entries.
# Change INSPECT_PATH to whichever layer you need to check.

INSPECT_PATH = os.path.join(VECTOR_SDE, r'SDE.Water\SDE.FEMA_Flood_Zone')

if arcpy.Exists(INSPECT_PATH):
    fields = pd.DataFrame(
        [{'name': f.name, 'alias': f.aliasName, 'type': f.type}
         for f in arcpy.ListFields(INSPECT_PATH)
         if f.type not in ('OID', 'Geometry')]
    )
    print(f'Fields in {os.path.basename(INSPECT_PATH)}:')
    display(fields)
else:
    print(f'Layer not found: {INSPECT_PATH}')